In [2]:
import torch
import torch.nn.functional as F
import torch_geometric
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn import GCNConv
import numpy as np
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

lam = 1e-3
epochs = 300
lr = 0.005

In [3]:
class SGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.conv2(x, edge_index)
        return x

def flatten_grads(grads):
    return torch.cat([g.contiguous().view(-1) for g in grads])

def train_and_get_loss(data, lmbda, seed=42):
    torch.manual_seed(seed)
    model = SGCN(data.num_features, 16, len(data.y.unique())).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[150, 250], gamma=0.1)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        reg = sum(p.pow(2.0).sum() for p in model.parameters())
        loss = criterion(out[data.train_mask], data.y[data.train_mask]) + lmbda * reg
        loss.backward()
        optimizer.step()
        scheduler.step()

    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        val_mask = data.val_mask 
        val_loss = criterion(out[val_mask], data.y[val_mask]).item()

    return model, val_loss

def get_node_gradients(model, data, mask, lmbda):
    model.eval()
    out = model(data.x, data.edge_index)
    params = list(model.parameters())
    criterion = torch.nn.CrossEntropyLoss()
    reg = sum(p.pow(2.0).sum() for p in model.parameters())
    
    grads_dict = {}
    indices = mask.nonzero(as_tuple=False).view(-1)
    
    for i in indices:
        i = i.item()
        l1 = criterion(out[i:i+1], data.y[i:i+1]) + lmbda * reg
        grad_z = torch.autograd.grad(l1, params, retain_graph=True)
        grads_dict[i] = flatten_grads(grad_z)
        
    return grads_dict

def create_noisy(data, noise_level, no_of_classes):
    data1 = data.clone()
    torch.manual_seed(42)
    rand_num = torch.rand(len(data.y), device=device)
    
    for k in range(no_of_classes):
        for j in range(no_of_classes - 1):
            prob_lower = (j) * noise_level / (no_of_classes - 1)
            prob_upper = (j + 1) * noise_level / (no_of_classes - 1)
            
            condition = (rand_num < prob_upper) & (prob_lower <= rand_num) & data.train_mask & (data.y == k)
            data1.y[condition] = (data1.y[condition] + (j + 1)) % no_of_classes
            
    return data1

In [4]:
def run_group_correlation_experiment(group_size):
    print("Starting Ground-Truth Correlation Test (Groups of ",group_size," Nodes)")
    dataset = Planetoid(root='data/py311/Planetoid', name='Cora', split="random", 
                        num_train_per_class=172, num_val=50, num_test=1000, 
                        transform=NormalizeFeatures())
    
    datap = dataset[0].clone().to(device)
    no_of_classes = len(datap.y.unique())
    data_bn = create_noisy(datap, 0.2, no_of_classes)

    # 1. Train Base Model
    print("1. Training Base Model...")
    base_model, base_val_loss = train_and_get_loss(data_bn, lam, seed=1)
    print(f"   Base Validation Loss: {base_val_loss:.6f}")

    # 2. Calculate Hessian & Predicted Influence
    print("2. Calculating Hessian and Predicted Influence...")
    base_model.eval()
    out = base_model(data_bn.x, data_bn.edge_index)
    reg = sum(p.pow(2.0).sum() for p in base_model.parameters())
    criterion = torch.nn.CrossEntropyLoss()
    
    l = criterion(out[data_bn.train_mask], data_bn.y[data_bn.train_mask]) + lam * reg
    grads = torch.autograd.grad(l, base_model.parameters(), create_graph=True, retain_graph=True)
    grad_flatten = flatten_grads(grads)
    
    hessian_tensor = torch.zeros(len(grad_flatten), len(grad_flatten), device=device)
    for i in range(len(grad_flatten)):
        grad_grad = torch.autograd.grad(grad_flatten[i], base_model.parameters(), retain_graph=True)
        hessian_tensor[i] = flatten_grads(grad_grad)
        
    hess_inv = torch.linalg.inv(hessian_tensor)
    del hessian_tensor
    torch.cuda.empty_cache()
    
    train_node_grads = get_node_gradients(base_model, data_bn, data_bn.train_mask, lam)
    
    val_mask = data_bn.val_mask 
    val_node_grads = get_node_gradients(base_model, data_bn, val_mask, lam)

    grad_test = torch.zeros(len(data_bn.y), len(grad_flatten), device=device)
    for i, grad_temp in val_node_grads.items():
        grad_test[i] = grad_temp

    # Get all training indices to pre-calculate individual influences
    train_indices = data_bn.train_mask.nonzero(as_tuple=False).view(-1).cpu().numpy()
    
    n_train = len(train_indices) # Number of training nodes
    n_val = val_mask.sum().item() # Number of validation nodes (50)
    
    print("3. Pre-calculating individual influences...")
    individual_pred_deltas = {}
    for node_idx in train_indices:
        grad_temp = train_node_grads[node_idx]
        

        infl_z = torch.matmul(hess_inv, grad_temp) / n_train
        
        pred_val_diffs = torch.matmul(grad_test, infl_z)
        pred_delta = pred_val_diffs.sum().item() / n_val
        
        individual_pred_deltas[node_idx] = pred_delta
    predicted_delta_loss = []
    actual_delta_loss = []

    
    np.random.seed(42)
    num_samples = 100
    
    sampled_groups = [np.random.choice(train_indices, group_size, replace=False) for _ in range(num_samples)]

    print(f"4. Retraining model {num_samples} times to get Ground Truth for groups...")
    for group in tqdm(sampled_groups):
        # PREDICTED 
        pred_delta_group = sum(individual_pred_deltas[idx] for idx in group)
        predicted_delta_loss.append(pred_delta_group)

        # ACTUAL
        data_mod = data_bn.clone()
        for node_idx in group:
            data_mod.train_mask[node_idx] = False
        
        group_tensor = torch.tensor(group, device=device)
        mask_src = torch.isin(data_mod.edge_index[0], group_tensor)
        mask_dst = torch.isin(data_mod.edge_index[1], group_tensor)
        edge_mask = ~(mask_src | mask_dst)
        data_mod.edge_index = data_mod.edge_index[:, edge_mask]

        _, mod_val_loss = train_and_get_loss(data_mod, lam, seed=1)
        
        actual_delta = mod_val_loss - base_val_loss
        actual_delta_loss.append(actual_delta)

    # 5. Output Run-by-Run Comparisons
    predicted_delta_loss = np.array(predicted_delta_loss)
    actual_delta_loss = np.array(actual_delta_loss)


    # 6. Correlation & Directional Analysis
    pearson_corr, _ = pearsonr(predicted_delta_loss, actual_delta_loss)
    spearman_corr, _ = spearmanr(predicted_delta_loss, actual_delta_loss)

    
    print("\n=== AGGREGATE STATS ===")
    print("Group Size: ",group_size)
    print(f"Pearson Correlation (Linear fit) : {pearson_corr:.4f}")
    print(f"Spearman Correlation (Rank fit)  : {spearman_corr:.4f}")

In [6]:
group_sizes=[1,2,25,50]
for i in group_sizes:
    run_group_correlation_experiment(i)

Starting Ground-Truth Correlation Test (Groups of  1  Nodes)
1. Training Base Model...
   Base Validation Loss: 1.306554
2. Calculating Hessian and Predicted Influence...
3. Pre-calculating individual influences...
4. Retraining model 100 times to get Ground Truth for groups...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:27<00:00,  3.70it/s]



=== AGGREGATE STATS ===
Group Size:  1
Pearson Correlation (Linear fit) : 0.8285
Spearman Correlation (Rank fit)  : 0.6878
Starting Ground-Truth Correlation Test (Groups of  2  Nodes)
1. Training Base Model...
   Base Validation Loss: 1.306554
2. Calculating Hessian and Predicted Influence...
3. Pre-calculating individual influences...
4. Retraining model 100 times to get Ground Truth for groups...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:27<00:00,  3.64it/s]



=== AGGREGATE STATS ===
Group Size:  2
Pearson Correlation (Linear fit) : 0.7707
Spearman Correlation (Rank fit)  : 0.6893
Starting Ground-Truth Correlation Test (Groups of  25  Nodes)
1. Training Base Model...
   Base Validation Loss: 1.306554
2. Calculating Hessian and Predicted Influence...
3. Pre-calculating individual influences...
4. Retraining model 100 times to get Ground Truth for groups...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:26<00:00,  3.71it/s]



=== AGGREGATE STATS ===
Group Size:  25
Pearson Correlation (Linear fit) : 0.7500
Spearman Correlation (Rank fit)  : 0.7605
Starting Ground-Truth Correlation Test (Groups of  50  Nodes)
1. Training Base Model...
   Base Validation Loss: 1.306554
2. Calculating Hessian and Predicted Influence...
3. Pre-calculating individual influences...
4. Retraining model 100 times to get Ground Truth for groups...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:27<00:00,  3.66it/s]


=== AGGREGATE STATS ===
Group Size:  50
Pearson Correlation (Linear fit) : 0.7670
Spearman Correlation (Rank fit)  : 0.7611


In [7]:
group_sizes=[200]
for i in group_sizes:
    run_group_correlation_experiment(i)

Starting Ground-Truth Correlation Test (Groups of  200  Nodes)
1. Training Base Model...
   Base Validation Loss: 1.306554
2. Calculating Hessian and Predicted Influence...
3. Pre-calculating individual influences...
4. Retraining model 100 times to get Ground Truth for groups...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:26<00:00,  3.73it/s]


=== AGGREGATE STATS ===
Group Size:  200
Pearson Correlation (Linear fit) : 0.7642
Spearman Correlation (Rank fit)  : 0.7344
